In [5]:
import pandas as pd
import numpy as np
import sklearn 
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_california_housing

In [6]:
# 1. Call the function to fetch the data object
housing = fetch_california_housing(as_frame = True) # as_frame for direct DataFrame
X = housing.data
Y = housing.target


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.20, random_state= 456 # last 3 id digits
)

In [11]:
scale = StandardScaler()

# Fit the scaler ONLY on the training data, then transform both sets
X_train_scaled = pd.DataFrame(
    scale.fit_transform(X_train), 
    columns=X_train.columns, 
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    scale.transform(X_test), 
    columns=X_test.columns, 
    index=X_test.index
)

In [13]:
X_train_scaled.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
1022,-0.746693,-0.922976,1.432652,1.590760,-0.957080,-0.075436,1.435423,-0.108500
7945,0.282415,1.225602,-0.084085,-0.224180,-0.386435,-0.000879,-0.823579,0.711551
3980,0.239169,0.429832,0.048203,-0.197956,-0.929738,-0.016605,-0.673604,0.476536
10689,-0.530571,-0.763822,-0.529645,-0.280413,-0.933266,-0.154423,-0.940747,0.921564
11510,0.419298,0.191101,-0.479132,0.017933,-0.758633,-0.110601,-0.884507,0.731552


## Training an MLP Regressor

In [59]:
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error

hidden_layer_sizes = (100, 50)
activation = "relu"
alpha = 1e-5
max_iter = 4000

mlp = MLPRegressor(
    hidden_layer_sizes=hidden_layer_sizes,
    activation=activation,
    alpha=alpha,
    random_state=0,
    max_iter=max_iter,
)

mlp.fit(X_train, y_train)

train_acc = mlp.score(X_train, y_train)
test_acc = mlp.score(X_test, y_test)

print("Train accuracy:", train_acc)
print("Test accuracy:", test_acc)

#evaluate using the Mean Squared Error
y_pred = mlp.predict(X_test)
print("MSE:", mean_squared_error(y_test, y_pred))

Train accuracy: -2.020972457427699
Test accuracy: -1.976550920565444
MSE: 4.076832415988575


## Comparing configurations

In [63]:
configs = [
    {"hidden": (50,),  "alpha": 1e-4},
    {"hidden": (100,), "alpha": 5e-5},
    {"hidden": (200,), "alpha": 1e-5},
]

for cfg in configs:
    model = MLPRegressor(hidden_layer_sizes=cfg["hidden"],
                          activation="relu",
                          alpha=cfg["alpha"],
                          random_state=0,
                          max_iter=10000)
    model.fit(X_train, y_train)
    tr = model.score(X_train, y_train)
    te = model.score(X_test, y_test)

    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    print(f"hidden={cfg['hidden']}, alpha={cfg['alpha']:>8g} | train={tr:.3f}, test={te:.3f} | MSE={mse:.4f}")

hidden=(50,), alpha=  0.0001 | train=0.281, test=0.384 | MSE=0.8435
hidden=(100,), alpha=   5e-05 | train=0.437, test=0.495 | MSE=0.6912
hidden=(200,), alpha=   1e-05 | train=0.308, test=0.345 | MSE=0.8977


#### <u>What does the above code show?<u>